# 03. Products Silver Layer: Data Exploration & Schema Validation
In this notebook, we perform the initial exploration of the Products dataset. This phase is crucial for identifying data quality issues, such as missing values, schema inconsistencies, and duplicate records, before moving to the cleaning and translation phase.

In [ ]:
# ── CELL 1: INITIALIZE SPARK SESSION ──────────────────────────────────────
import os
import pyspark
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Build SparkSession locally to avoid Python version mismatch with external cluster
spark = SparkSession.builder \
    .appName("olist-notebook-analysis") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to reduce noise inside the notebook
spark.sparkContext.setLogLevel("WARN")

print("Spark Session created successfully!")
print("Spark Master URI:", spark.sparkContext.master)

🎯 Spark Session created successfully!
Spark Master URI: local[*]


In [2]:
# 1. Load the Products dataset from the Bronze layer in MinIO
# Delta format automatically preserves exact column data types (no inferSchema needed)
df_products = (
    spark.read
    .format("delta")
    .load("s3a://bronze/csv/products/")
)

# 2. Display the first 10 rows inside the notebook
display(df_products.limit(10))

DataFrame[product_id: string, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, _ingested_at: timestamp, _source_file: string]

### 1. Schema Inspection
Checking the inferred schema ensures that data types (e.g., weights, dimensions, and IDs) are correctly interpreted by Spark for subsequent transformations.

In [3]:
# Review the data types and column names
df_products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Schema Enforcement
Cast all product attributes to their target data types to ensure consistent schema representation and data quality.

In [4]:
from pyspark.sql.types import DecimalType, IntegerType, StringType
from pyspark.sql import functions as F

# Explicitly cast to target data types
df_products = df_products \
    .withColumn("product_id", F.col("product_id").cast(StringType())) \
    .withColumn("product_category_name", F.col("product_category_name").cast(StringType())) \
    .withColumn("product_name_lenght", F.col("product_name_lenght").cast(IntegerType())) \
    .withColumn("product_description_lenght", F.col("product_description_lenght").cast(IntegerType())) \
    .withColumn("product_photos_qty", F.col("product_photos_qty").cast(IntegerType())) \
    .withColumn("product_weight_g", F.col("product_weight_g").cast(DecimalType(10, 2))) \
    .withColumn("product_length_cm", F.col("product_length_cm").cast(DecimalType(10, 2))) \
    .withColumn("product_height_cm", F.col("product_height_cm").cast(DecimalType(10, 2))) \
    .withColumn("product_width_cm", F.col("product_width_cm").cast(DecimalType(10, 2)))

# Verify schema
print("=== Final Schema Verification ===")
df_products.printSchema()

=== Final Schema Verification ===
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: decimal(10,2) (nullable = true)
 |-- product_length_cm: decimal(10,2) (nullable = true)
 |-- product_height_cm: decimal(10,2) (nullable = true)
 |-- product_width_cm: decimal(10,2) (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



### 2. Statistical Summary
Generating descriptive statistics helps in detecting outliers or logical errors in physical attributes like product weight and dimensions.

In [5]:
display(df_products.describe())

DataFrame[summary: string, product_id: string, product_category_name: string, product_name_lenght: string, product_description_lenght: string, product_photos_qty: string, product_weight_g: string, product_length_cm: string, product_height_cm: string, product_width_cm: string, _source_file: string]

### 3. Duplicate Record Analysis
Identifying exact duplicate rows to ensure each product record is unique within the dataset.

In [6]:
from pyspark.sql import functions as F

# 1. Define the initial audit dataframe 
# We create it by adding the audit columns to the existing df_products structure
# and then filtering for an empty result to keep the schema without data
item_errors_audit = df_products.withColumn("error_reason", F.lit(None).cast("string")) \
                               .withColumn("error_detected_at", F.current_timestamp()) \
                               .filter(F.lit(False))

# 2. Identify duplicates
total_count = df_products.count()
distinct_df = df_products.distinct()
distinct_count = distinct_df.count()

duplicate_count = total_count - distinct_count

# 3. Process duplicates if found
if duplicate_count > 0:
    print(f"Duplicates found: {duplicate_count}")
    
    # Isolate the duplicated rows and add audit info
    duplicates_df = df_products.subtract(distinct_df) \
        .withColumn("error_reason", F.lit("Duplicate record found - Keeping only one instance")) \
        .withColumn("error_detected_at", F.current_timestamp())
    
    # Append duplicates to the audit log
    item_errors_audit = item_errors_audit.unionByName(duplicates_df, allowMissingColumns=True)
    
    # Update the main dataframe to contain only unique records
    df_products = distinct_df
    
    print("Duplicates captured in audit log and removed from main DataFrame.")
else:
    print("No duplicates found. Data is clean.")

No duplicates found. Data is clean.


### 4. Null Value Profiling
Quantifying missing values across all columns to pinpoint which features (e.g., categories or descriptions) require imputation or filtering.

In [7]:
from pyspark.sql import functions as F
null_counts = df_products.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_products.columns])

display(null_counts)

DataFrame[product_id: bigint, product_category_name: bigint, product_name_lenght: bigint, product_description_lenght: bigint, product_photos_qty: bigint, product_weight_g: bigint, product_length_cm: bigint, product_height_cm: bigint, product_width_cm: bigint, _ingested_at: bigint, _source_file: bigint]

### 5. Cross-Layer Impact Analysis: Missing Attributes vs. Sales
To evaluate the business impact of products with missing metadata (names, descriptions, or categories), we perform a **Left Join** with our existing `silver_order_items` table. This allows us to quantify the total revenue and order frequency linked to incomplete product records.

In [8]:
# Read the refined order items table directly from the Silver layer on MinIO
items_silver = (
    spark.read
    .format("delta")
    .load("s3a://silver/refined/order_items/")
)

from pyspark.sql import functions as F

# 1. Filter products with any missing descriptive metadata
null_products = df_products.filter(
    F.col("product_name_lenght").isNull() | 
    F.col("product_description_lenght").isNull() | 
    F.col("product_photos_qty").isNull() |
    F.col("product_category_name").isNull()
).select(
    "product_id", 
    "product_category_name", 
    "product_name_lenght", 
    "product_description_lenght", 
    "product_photos_qty"
)

# 2. Join with sales data to assess financial impact
null_products_sales = null_products.join(items_silver, on="product_id", how="left")

# 3. Aggregate sales metrics per incomplete product
sales_summary = null_products_sales.groupBy(
    "product_id", 
    "product_category_name", 
    "product_name_lenght", 
    "product_description_lenght", 
    "product_photos_qty"
).agg(
    F.count("order_id").alias("times_bought"),
    F.round(F.sum("price"), 2).alias("total_revenue")
).orderBy(F.desc("times_bought"))

sales_summary.show()

+--------------------+---------------------+-------------------+--------------------------+------------------+------------+-------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|times_bought|total_revenue|
+--------------------+---------------------+-------------------+--------------------------+------------------+------------+-------------+
|5a848e4ab52fd5445...|                 NULL|               NULL|                      NULL|              NULL|         197|     24229.03|
|b1d207586fca400a2...|                 NULL|               NULL|                      NULL|              NULL|          48|      7152.00|
|76d1a1a9d21ab677a...|                 NULL|               NULL|                      NULL|              NULL|          32|      5712.64|
|3b60d513e90300a4e...|                 NULL|               NULL|                      NULL|              NULL|          29|      4393.33|
|ad88641611c35ebd5...|            

### 6. Data Loss Percentage Profiling
We calculate the exact percentage of missing values for each critical column. This metric helps in determining the "Data Health Score" for the Products domain.

In [9]:
from pyspark.sql import functions as F

# 1. Total record count for ratio calculation
total_count = df_products.count()

# 2. Identify null counts for descriptive attributes
null_analysis = df_products.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(f"{c}_null_count") 
    for c in ["product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty"]
])

# 3. Calculate and display loss percentages
print(f"Total Products: {total_count}")
print("-" * 30)

null_analysis.select([
    F.round((F.col(f"{c}_null_count") / total_count) * 100, 2).alias(f"{c}_loss_percentage")
    for c in ["product_category_name", "product_name_lenght", "product_description_lenght", "product_photos_qty"]
]).show()

Total Products: 32951
------------------------------
+-------------------------------------+-----------------------------------+------------------------------------------+----------------------------------+
|product_category_name_loss_percentage|product_name_lenght_loss_percentage|product_description_lenght_loss_percentage|product_photos_qty_loss_percentage|
+-------------------------------------+-----------------------------------+------------------------------------------+----------------------------------+
|                                 1.85|                               1.85|                                      1.85|                              1.85|
+-------------------------------------+-----------------------------------+------------------------------------------+----------------------------------+



### 7. Physical Dimensions Integrity Check
Products missing physical dimensions (weight, length, etc.) pose a risk for logistics and shipping calculations. We isolate these records and cross-reference them with sales data to identify active products with missing logistics data.

In [10]:
from pyspark.sql import functions as F

# 1. Identify product IDs missing physical dimensions
troubled_product_ids = df_products.filter(
    F.col("product_weight_g").isNull() | 
    F.col("product_length_cm").isNull() | 
    F.col("product_height_cm").isNull() | 
    F.col("product_width_cm").isNull()
).select("product_id")

# 2. Join with sales data using an Inner Join to focus on active products
troubled_sales_report = troubled_product_ids.join(items_silver, on="product_id", how="inner")

# 3. Generate summary of sales and associated sellers for these records
final_report = troubled_sales_report.groupBy("product_id").agg(
    F.count("order_id").alias("times_bought"),
    F.round(F.sum("price"), 2).alias("total_revenue"),
    F.collect_set("seller_id").alias("sellers_list") 
)

print("Sales report for dimensionless products:")
final_report.show(truncate=False)

Sales report for dimensionless products:
+--------------------------------+------------+-------------+----------------------------------+
|product_id                      |times_bought|total_revenue|sellers_list                      |
+--------------------------------+------------+-------------+----------------------------------+
|09ff539a621711667c43eba6a3bd8466|1           |1934.00      |[8b8cfc8305aa441e4239358c9f6f2485]|
|5eb564652db742ff8f28759cd8d2652a|17          |563.00       |[4e922959ae960d389249c378d1c939f5]|
+--------------------------------+------------+-------------+----------------------------------+



@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Data Quality Audit & Metadata Imputation
Identify records with missing physical attributes or descriptive metadata, log them for audit, and impute missing values using category-based median statistics or defined defaults.

In [11]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

# 1. Capture errors and log to the main 'item_errors_audit' DataFrame
missing_cond = (
    F.col("product_weight_g").isNull() | 
    F.col("product_length_cm").isNull() | 
    F.col("product_height_cm").isNull() | 
    F.col("product_width_cm").isNull() |
    F.col("product_category_name").isNull() |
    F.col("product_name_lenght").isNull() |
    F.col("product_description_lenght").isNull() |
    F.col("product_photos_qty").isNull()
)

# Capture records to audit
errors_df = df_products.filter(missing_cond) \
    .withColumn("error_reason", F.lit("Missing physical dimensions, weight, or descriptive metadata")) \
    .withColumn("error_detected_at", F.current_timestamp())

# Union with the master audit log
item_errors_audit = item_errors_audit.unionByName(errors_df, allowMissingColumns=True)

# 2. Imputation Path
window_category = Window.partitionBy("product_category_name")

# Create products_cleaned from df_products
products_cleaned = df_products \
    .withColumn("product_weight_g", F.coalesce(F.col("product_weight_g"), F.percentile_approx("product_weight_g", 0.5).over(window_category)).cast(DecimalType(10, 2))) \
    .withColumn("product_length_cm", F.coalesce(F.col("product_length_cm"), F.percentile_approx("product_length_cm", 0.5).over(window_category)).cast(DecimalType(10, 2))) \
    .withColumn("product_height_cm", F.coalesce(F.col("product_height_cm"), F.percentile_approx("product_height_cm", 0.5).over(window_category)).cast(DecimalType(10, 2))) \
    .withColumn("product_width_cm", F.coalesce(F.col("product_width_cm"), F.percentile_approx("product_width_cm", 0.5).over(window_category)).cast(DecimalType(10, 2)))

# 3. Final Fallback Imputation
descriptive_defaults = {
    "product_category_name": "Unknown Category",
    "product_name_lenght": 0,
    "product_description_lenght": 0,
    "product_photos_qty": 0
}

products_cleaned = products_cleaned.fillna(descriptive_defaults) \
    .fillna(-1.00, subset=["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"])

print(f"Total records logged in audit: {errors_df.count()}")
print(f"products_cleaned is now cleaned and ready for the next step.")

Total records logged in audit: 611
products_cleaned is now cleaned and ready for the next step.


In [12]:
from pyspark.sql import functions as F
null_counts = products_cleaned.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in products_cleaned.columns])

display(null_counts)

DataFrame[product_id: bigint, product_category_name: bigint, product_name_lenght: bigint, product_description_lenght: bigint, product_photos_qty: bigint, product_weight_g: bigint, product_length_cm: bigint, product_height_cm: bigint, product_width_cm: bigint, _ingested_at: bigint, _source_file: bigint]

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Product Volume Calculation
Compute the volumetric size of products based on valid physical dimensions, while enforcing a defensive '-1.00' value for missing or incomplete data.

In [13]:
from pyspark.sql import functions as F

# Calculate volume only if dimensions are valid (not -1.00)
# Otherwise, set to -1.00 to indicate that size could not be calculated
products_cleaned = products_cleaned.withColumn(
    "product_size_cm3",
    F.when(
        (F.col("product_length_cm") != -1.00) & 
        (F.col("product_height_cm") != -1.00) & 
        (F.col("product_width_cm") != -1.00),
        F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm")
    ).otherwise(F.lit(-1.00).cast("decimal(10,2)"))
)

# Verify
print("=== VERIFYING PRODUCT SIZE CALCULATION ===")
products_cleaned.select(
    "product_id", 
    "product_length_cm", 
    "product_height_cm", 
    "product_width_cm", 
    "product_size_cm3"
).show(5, truncate=False)

=== VERIFYING PRODUCT SIZE CALCULATION ===
+--------------------------------+-----------------+-----------------+----------------+----------------+
|product_id                      |product_length_cm|product_height_cm|product_width_cm|product_size_cm3|
+--------------------------------+-----------------+-----------------+----------------+----------------+
|a41e356c76fab66334f36de622ecbd3a|17.00            |14.00            |12.00           |2856.000000     |
|d8dee61c2034d6d075997acef1870e9b|16.00            |7.00             |20.00           |2240.000000     |
|56139431d72cd51f19eb9f7dae4d1617|20.00            |20.00            |20.00           |8000.000000     |
|46b48281eb6d663ced748f324108c733|41.00            |30.00            |41.00           |50430.000000    |
|5fb61f482620cb672f5e586bb132eae9|35.00            |7.00             |12.00           |2940.000000     |
+--------------------------------+-----------------+-----------------+----------------+----------------+
only showing

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Logistics Categorization
Classify products into size and weight tiers based on computed volume and recorded weight to optimize fulfillment and shipping strategies.

In [14]:
from pyspark.sql import functions as F

# Add Logistical Size and Weight Categories
products_cleaned = products_cleaned.withColumn(
    "logistics_size_category",
    F.when(F.col("product_size_cm3") <= 0, "Unknown Size") # Handling our -1.00 or missing values
     .when(F.col("product_size_cm3") <= 5000, "Small Box")
     .when(F.col("product_size_cm3") <= 20000, "Medium Box")
     .otherwise("Large Parcel")
).withColumn(
    "logistics_weight_category",
    F.when(F.col("product_weight_g") <= 0, "Unknown Weight") # Handling our -1.00 or missing values
     .when(F.col("product_weight_g") <= 2000, "Lightweight")
     .when(F.col("product_weight_g") <= 10000, "Midweight")
     .otherwise("Heavyweight")
)

# Display verification
print("=== VERIFYING LOGISTICS CATEGORIZATION ===")
products_cleaned.select(
    "product_id", 
    "product_size_cm3", 
    "logistics_size_category", 
    "product_weight_g", 
    "logistics_weight_category"
).show(10, truncate=False)

=== VERIFYING LOGISTICS CATEGORIZATION ===
+--------------------------------+----------------+-----------------------+----------------+-------------------------+
|product_id                      |product_size_cm3|logistics_size_category|product_weight_g|logistics_weight_category|
+--------------------------------+----------------+-----------------------+----------------+-------------------------+
|a41e356c76fab66334f36de622ecbd3a|2856.000000     |Small Box              |650.00          |Lightweight              |
|d8dee61c2034d6d075997acef1870e9b|2240.000000     |Small Box              |300.00          |Lightweight              |
|56139431d72cd51f19eb9f7dae4d1617|8000.000000     |Medium Box             |200.00          |Lightweight              |
|46b48281eb6d663ced748f324108c733|50430.000000    |Large Parcel           |18500.00        |Heavyweight              |
|5fb61f482620cb672f5e586bb132eae9|2940.000000     |Small Box              |300.00          |Lightweight              |
|e107

### 10. Category Translation Ingestion
To provide English-speaking stakeholders with readable product insights, we load the official category translation mapping. This table serves as the primary lookup for converting Portuguese category names to English.

In [16]:
# 1. Load the category translation dataset from the Bronze layer in MinIO
# FIXED: Using the correct folder name 'category_translation' generated by the Ingestion script
df_translation = (
    spark.read
    .format("delta")
    .load("s3a://bronze/csv/category_translation/")
)

# 2. Display the first 10 rows inside the notebook
display(df_translation.limit(10))

DataFrame[product_category_name: string, product_category_name_english: string, _ingested_at: timestamp, _source_file: string]

### 11. Translation Table Integrity Check
Before using the translation table as a reference, we must ensure it is free of null values and duplicates that could lead to data loss or ambiguity during the join.

In [17]:
print("Checking for nulls or duplicates in translation table:")
df_translation.select(
    F.count(F.when(F.col("product_category_name").isNull(), True)).alias("null_portuguese"),
    F.count(F.when(F.col("product_category_name_english").isNull(), True)).alias("null_english")
).show()

Checking for nulls or duplicates in translation table:
+---------------+------------+
|null_portuguese|null_english|
+---------------+------------+
|              0|           0|
+---------------+------------+



### 12. Identifying Unmapped Categories
We perform a **Left Anti Join** between our cleaned products and the translation table. This identifies specific Portuguese categories that exist in our inventory but lack a corresponding English translation in the lookup table.

In [18]:
# Identify categories in Products that are missing from the Translation table
# We use Left Anti Join to find 'product_category_name' values that don't exist in df_translations
missing_translations = products_cleaned.select("product_category_name").distinct() \
    .join(df_translation, "product_category_name", "left_anti")

# Log the number of untranslated categories
print(f"Number of categories missing English translation: {missing_translations.count()}")
display(missing_translations)

Number of categories missing English translation: 3


DataFrame[product_category_name: string]

### 13. Frequency Analysis of Untranslated Categories
Quantifying how many products are affected by missing translations. This helps prioritize whether manual mapping is required based on the volume of records involved.

In [19]:
# Count rows for each untranslated category value
anomalies_counts = products_cleaned.filter(
    F.col("product_category_name").isin("pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos", "unknown")
).groupBy("product_category_name").count()

display(anomalies_counts)

DataFrame[product_category_name: string, count: bigint]

### 14. Synonym and Keyword Search
We search the existing translation table for keywords like "kitchen", "game", or "food". This ensures that the missing categories are truly unique and not just variations of existing English terms already present in the system.

In [20]:
# Search for related keywords in the current translation table
search_keywords = df_translation.filter(
    F.col("product_category_name_english").contains("kitchen") | 
    F.col("product_category_name_english").contains("game") |
    F.col("product_category_name_english").contains("food")
)

search_keywords.show()

+---------------------+-----------------------------+--------------------+--------------------+
|product_category_name|product_category_name_english|        _ingested_at|        _source_file|
+---------------------+-----------------------------+--------------------+--------------------+
|    alimentos_bebidas|                   food_drink|2026-06-08 03:38:...|product_category_...|
|       consoles_games|               consoles_games|2026-06-08 03:38:...|product_category_...|
| moveis_cozinha_ar...|         kitchen_dining_la...|2026-06-08 03:38:...|product_category_...|
|            alimentos|                         food|2026-06-08 03:38:...|product_category_...|
+---------------------+-----------------------------+--------------------+--------------------+



### 15. Category Translation Logic
To finalize the product categorization, we apply a three-tier translation strategy:
1. **Manual Mapping:** Handle specific known missing terms (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`).
2. **Lookup Join:** Retrieve translations from the official translation table.
3. **Fallback:** Assign any remaining nulls to "unknown" to ensure 100% column population.

In [21]:
from pyspark.sql import functions as F

# 1. Apply manual mapping for the specific identified anomalies
products_with_manual = products_cleaned.withColumn(
    "manual_translation",
    F.when(F.col("product_category_name") == "pc_gamer", "pc_gamer")
     .when(F.col("product_category_name") == "portateis_cozinha_e_preparadores_de_alimentos", "kitchen_portables_and_food_preparers")
     .otherwise(None)
)

# 2. Join with the official translation table
products_joined = products_with_manual.join(df_translation, on="product_category_name", how="left")

# 3. Consolidate translations: Manual > Table > Fallback (unknown)
products_final = products_joined.withColumn(
    "product_category_name_english",
    F.coalesce(
        F.col("manual_translation"), 
        F.col("product_category_name_english"), 
        F.lit("unknown")
    )
).drop("manual_translation") # Remove helper column

# Preview final translation results
products_final.select("product_category_name", "product_category_name_english").distinct().show(20)

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|           perfumaria|                    perfumery|
| instrumentos_musi...|          musical_instruments|
|           brinquedos|                         toys|
|           automotivo|                         auto|
|            telefonia|                    telephony|
|                  pcs|                    computers|
|              bebidas|                       drinks|
|               musica|                        music|
| tablets_impressao...|         tablets_printing_...|
|     malas_acessorios|          luggage_accessories|
|    moveis_escritorio|             office_furniture|
|      eletroportateis|             small_appliances|
|         climatizacao|             air_conditioning|
| construcao_ferram...|         construction_tool...|
|           la_cuisine|                   la_cuisine|
| fashion_roupa_inf...|     

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Unknown Category Auditing
Identify and log products with undefined categories into the master audit system to maintain data quality traceability and support future enrichment efforts.

In [23]:
from pyspark.sql import functions as F

# 1. Filter and select ONLY clean, specific columns to completely drop any bronze metadata like _ingested_at
unknown_categories_df = products_final.filter(F.col("product_category_name") == "Unknown Category") \
    .select(
        F.col("product_id").cast("string"),
        F.col("product_category_name").cast("string").alias("failed_value"),
        F.lit("Product category is unknown").alias("error_reason"),
        F.current_timestamp().alias("error_detected_at")
    )

# 2. Append safely to the master audit log without any column duplication or conflict
item_errors_audit = item_errors_audit.unionByName(unknown_categories_df, allowMissingColumns=True)

print(f"Total 'unknown' category rows appended to error table: {unknown_categories_df.count()}")

Total 'unknown' category rows appended to error table: 610


In [24]:
display(products_final.limit(10))

DataFrame[product_category_name: string, product_id: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: decimal(10,2), product_length_cm: decimal(10,2), product_height_cm: decimal(10,2), product_width_cm: decimal(10,2), _ingested_at: timestamp, _source_file: string, product_size_cm3: decimal(32,6), logistics_size_category: string, logistics_weight_category: string, product_category_name_english: string, _ingested_at: timestamp, _source_file: string]

### 17. Schema Refinement: Dropping Original Category and Renaming English Column**
To streamline the schema and enforce English as the standard language for product categories, we drop the original `product_category_name` column and rename `product_category_name_english` to take its place within the `products_final` DataFrame.

In [25]:
from pyspark.sql import functions as F

# Drop the original category column and rename the English column to take its place
products_final = products_final \
    .drop("product_category_name") \
    .withColumnRenamed("product_category_name_english", "product_category_name")

# Verify the final schema and column names
print("=== VERIFYING FINAL SCHEMA COLUMNS ===")
products_final.printSchema()

# Show a quick sample to ensure data is correct
products_final.select("product_id", "product_category_name").show(5, truncate=False)

=== VERIFYING FINAL SCHEMA COLUMNS ===
root
 |-- product_id: string (nullable = true)
 |-- product_name_lenght: integer (nullable = false)
 |-- product_description_lenght: integer (nullable = false)
 |-- product_photos_qty: integer (nullable = false)
 |-- product_weight_g: decimal(10,2) (nullable = true)
 |-- product_length_cm: decimal(10,2) (nullable = true)
 |-- product_height_cm: decimal(10,2) (nullable = true)
 |-- product_width_cm: decimal(10,2) (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- product_size_cm3: decimal(32,6) (nullable = true)
 |-- logistics_size_category: string (nullable = false)
 |-- logistics_weight_category: string (nullable = false)
 |-- product_category_name: string (nullable = false)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

+--------------------------------+---------------------+
|product_id                      |product_category_name|
+------

In [28]:
from pyspark.sql import functions as F

# 1. Get all column names
all_cols = products_final.columns

# Find duplicate columns (like _ingested_at)
duplicate_cols = set([c for c in all_cols if all_cols.count(c) > 1])

# Re-create the dataframe by dropping the duplicate columns completely from the reference
# This approach forces Spark to resolve the ambiguity by eliminating the troublemakers
df_deduplicated = products_final
for col_to_drop in duplicate_cols:
    # This drops ALL instances of the ambiguous column to clean the slate
    df_deduplicated = df_deduplicated.drop(col_to_drop)

# 2. Run the Null counting loop safely on the remaining clear columns
null_counts = df_deduplicated.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_deduplicated.columns
])

# 3. Display the final counts
display(null_counts)

DataFrame[product_id: bigint, product_name_lenght: bigint, product_description_lenght: bigint, product_photos_qty: bigint, product_weight_g: bigint, product_length_cm: bigint, product_height_cm: bigint, product_width_cm: bigint, product_size_cm3: bigint, logistics_size_category: bigint, logistics_weight_category: bigint, product_category_name: bigint]

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Schema Enforcement via Mapping
Apply a standardized schema to the final products DataFrame using a dynamic mapping function to ensure data type consistency and operational readiness.

In [29]:
from pyspark.sql.types import StringType, IntegerType, DecimalType
import pyspark.sql.functions as F

# 1. Define the target schema mapping
target_schema = {
    "product_id": StringType(),
    "product_category_name": StringType(),
    "product_name_lenght": IntegerType(),
    "product_description_lenght": IntegerType(),
    "product_photos_qty": IntegerType(),
    "product_weight_g": DecimalType(10, 2),
    "product_length_cm": DecimalType(10, 2),
    "product_height_cm": DecimalType(10, 2),
    "product_width_cm": DecimalType(10, 2),
    "product_size_cm3": DecimalType(10, 2),
    "logistics_size_category": StringType(),
    "logistics_weight_category": StringType()
}

# 2. Force cast function
def enforce_schema(df, target_schema):
    print("=== Enforcing Schema: Casting all columns to target types ===")
    
    # Iterate through the schema and apply cast
    for col_name, data_type in target_schema.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, F.col(col_name).cast(data_type))
        else:
            print(f"Warning: Column '{col_name}' not found in DataFrame, skipping...")
            
    print("✅ Schema enforcement complete.")
    return df

# Execute enforcement
products_final = enforce_schema(products_final, target_schema)

# Optional: Verify final types
products_final.printSchema()

=== Enforcing Schema: Casting all columns to target types ===
✅ Schema enforcement complete.
root
 |-- product_id: string (nullable = true)
 |-- product_name_lenght: integer (nullable = false)
 |-- product_description_lenght: integer (nullable = false)
 |-- product_photos_qty: integer (nullable = false)
 |-- product_weight_g: decimal(10,2) (nullable = true)
 |-- product_length_cm: decimal(10,2) (nullable = true)
 |-- product_height_cm: decimal(10,2) (nullable = true)
 |-- product_width_cm: decimal(10,2) (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- product_size_cm3: decimal(10,2) (nullable = true)
 |-- logistics_size_category: string (nullable = false)
 |-- logistics_weight_category: string (nullable = false)
 |-- product_category_name: string (nullable = false)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Final Audit Persistence
Persist the product-level data quality audit log to the Lakehouse in Delta format, ensuring pipeline stability and historical traceability of anomalies.

In [30]:
# ==========================================================================
# FINAL PERSISTENCE: SAVING PRODUCTS/ITEMS QUALITY AUDIT LOGS
# ==========================================================================

# Define the absolute MinIO S3A storage path for the products audit logs
PRODUCTS_AUDIT_LOG_PATH = "s3a://silver/qa_issues/silver_products_errors/"

print(f"Saving data quality audit logs to MinIO path: {PRODUCTS_AUDIT_LOG_PATH}...")

# Save the audit dataframe using Delta format with overwrite mode (FIXED: replaced saveAsTable)
item_errors_audit.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(PRODUCTS_AUDIT_LOG_PATH)

# Explicitly refresh the Delta cache for this path to ensure instant data governance visibility
spark.catalog.refreshByPath(PRODUCTS_AUDIT_LOG_PATH)

print(f"Successfully saved {item_errors_audit.count()} error records to: {PRODUCTS_AUDIT_LOG_PATH}")

Saving data quality audit logs to MinIO path: s3a://silver/qa_issues/silver_products_errors/...
Successfully saved 1221 error records to: s3a://silver/qa_issues/silver_products_errors/


In [33]:
products_final = products_final.drop("_ingested_at", "_source_file")
print("Cleaned Columns:", products_final.columns)

Cleaned Columns: ['product_id', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_size_cm3', 'logistics_size_category', 'logistics_weight_category', 'product_category_name']


### 18. Final Silver Products Persistence
With the schema refined, dimensions validated, and categories translated, we save the final dataset as a Delta table and export a Parquet version for external compatibility.

In [34]:
# ==========================================================================
# FINAL PERSISTENCE: SAVING REFINED SILVER PRODUCTS
# ==========================================================================

# Define the target absolute storage paths on MinIO
SILVER_PRODUCTS_DELTA_PATH   = "s3a://silver/refined/products/"
SILVER_PRODUCTS_PARQUET_PATH = "s3a://silver/refined/products_parquet/"

# 1. Save as a Delta Table using direct MinIO S3A paths (FIXED: replaced saveAsTable)
products_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_PRODUCTS_DELTA_PATH)

# Refresh the Delta cache for immediate query capability inside the cluster
spark.catalog.refreshByPath(SILVER_PRODUCTS_DELTA_PATH)


# 2. Export as Parquet files to the Silver directory on MinIO (FIXED: updated local path to S3A)
# Coalesce(1) consolidates output into a single partition for easier downstream distribution
products_final.coalesce(1).write \
    .mode("overwrite") \
    .parquet(SILVER_PRODUCTS_PARQUET_PATH)


# Final confirmation log
print("Success! Silver Products table is secured and safely saved to MinIO as Delta and Parquet.")

Success! Silver Products table is secured and safely saved to MinIO as Delta and Parquet.
